# 05. Tree Models Without Exogenous Variables

외생변수를 제외한 트리 기반 모델 실험 노트북입니다.

- SARIMA, RandomForest, XGBoost, LightGBM 비교
- GridSearchCV 기반 하이퍼파라미터 탐색
- 단일 클러스터 기준 RF MAPE 3.36% 등 트리 모델 잠재력 확인

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import prophet

/home/osanie/.conda/envs/jsys/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
final = pd.read_csv('../data/final.csv', encoding='utf-8-sig')
final

,판매월,제조업체,상품명,판매수량,대분류,중분류,한파,폭염,호우,생활물가지수,음식료품_계절조정지수
0,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
1,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,2.0,면류.라면류,면류,22,0,0,101.20,105.5
2,2021-01,88030600.0,(백제물산)즉석쌀국수멸치맛92g,1.0,면류.라면류,면류,22,0,0,101.20,105.5
3,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,2.0,면류.라면류,라면류,22,0,0,101.20,105.5
4,2021-01,88010455.0,(오뚜기)열라면종이(용기)105g,1.0,면류.라면류,라면류,22,0,0,101.20,105.5
...,...,...,...,...,...,...,...,...,...,...,...
80578,2024-02,NaN,NaN,NaN,NaN,NaN,2,0,12,116.32,91.0
80579,2024-03,NaN,NaN,NaN,NaN,NaN,5,0,6,116.62,94.5
80580,2024-04,NaN,NaN,NaN,NaN,NaN,0,0,0,116.59,94.0
80581,2024-05,NaN,NaN,NaN,NaN,NaN,0,0,21,116.53,94.6


In [6]:
final_1 = final.copy()
final_1.drop(columns=['상품명', '대분류', '중분류', '한파', '폭염', '호우', '생활물가지수', '음식료품_계절조정지수'], inplace=True)

In [7]:
# 2024년 데이터 삭제
final_1['판매월'] = pd.to_datetime(final_1['판매월'], format='%Y-%m')
final_1 = final_1[final_1['판매월'].dt.year != 2024]

---

## 제조업체 군집화 예측

#### 1. 제조업체를 동일한 분포를 띄는 네 개의 그룹으로 분리 (A, B, C, D)
    1-1. 각각은 월별로 분포를 찍어봤을 때에도 분포가 비슷하면 좋음.
#### 2. 해당 제조업체들을 A, B, C, D로 치환하여 학습
#### 3. 예측도 네 개로 나눠서 진행. 월별로 네 개의 제조업체를 합쳐서 MAPE 계산

In [8]:
import pandas as pd
import numpy as np

# 제조업체별 판매수량 합계 계산 및 내림차순 정렬
total_sales_by_manufacturer = final_1.groupby('제조업체')['판매수량'].sum().reset_index().sort_values(by='판매수량', ascending=False).reset_index(drop=True)

# 목표 군집 수
n_clusters = 4

# 총 판매수량을 기준으로 균등하게 군집화
total_sales = total_sales_by_manufacturer['판매수량'].sum()  # 전체 판매수량 합계
target_sales_per_cluster = total_sales / n_clusters  # 각 군집이 목표로 하는 판매수량 합계

# 그리디 알고리즘을 사용하여 제조업체를 군집화
clusters = [[] for _ in range(n_clusters)]  # 각 군집을 담을 리스트
cluster_sales = [0] * n_clusters  # 각 군집의 현재 판매수량 합계
cluster_labels = []  # 제조업체에 대한 군집 레이블

for idx, row in total_sales_by_manufacturer.iterrows():
    # 가장 적은 판매수량을 가진 군집에 제조업체를 추가
    min_cluster_idx = np.argmin(cluster_sales)
    clusters[min_cluster_idx].append(row['제조업체'])
    cluster_sales[min_cluster_idx] += row['판매수량']
    cluster_labels.append(min_cluster_idx + 1)  # 군집 번호는 1부터 시작

# 결과를 데이터프레임으로 변환
total_sales_by_manufacturer['군집'] = cluster_labels

# 각 군집별 총 판매수량 출력
for i in range(n_clusters):
    cluster_total_sales = total_sales_by_manufacturer[total_sales_by_manufacturer['군집'] == i+1]['판매수량'].sum()
    print(f"Cluster {i+1} (총 판매수량: {cluster_total_sales})")

# 최종 결과 데이터프레임 출력
total_sales_by_manufacturer

Cluster 1 (총 판매수량: 106999.0)
Cluster 2 (총 판매수량: 69525.0)
Cluster 3 (총 판매수량: 39133.0)
Cluster 4 (총 판매수량: 39129.0)


,제조업체,판매수량,군집
0,88010430.0,106999.0,1
1,88010455.0,69525.0,2
2,88030600.0,26296.0,3
3,88010454.0,13272.0,4
4,88010731.0,9620.0,4
5,88011285.0,7499.0,4
6,88010453.0,6270.0,3
7,88010732.0,4617.0,4
8,88010438.0,3838.0,3
9,88096952.0,1796.0,4


In [9]:
final_1['cluster'] = final_1['제조업체'].map(dict(zip(total_sales_by_manufacturer['제조업체'], total_sales_by_manufacturer['군집'])))

In [10]:
final_1 = final_1.drop(columns=['제조업체'])

In [11]:
final_1

,판매월,판매수량,cluster
0,2021-01-01,1.0,2
1,2021-01-01,2.0,3
2,2021-01-01,1.0,3
3,2021-01-01,2.0,2
4,2021-01-01,1.0,2
...,...,...,...
80572,2023-12-01,1.0,1
80573,2023-12-01,3.0,2
80574,2023-12-01,10.0,3
80575,2023-12-01,1.0,1


In [12]:
import pandas as pd

# 데이터 로드 (이미 데이터프레임으로 불러온 경우 이 부분 생략)
# df = pd.read_csv('your_data.csv')

# '판매월'을 datetime 형식으로 변환
final_1['판매월'] = pd.to_datetime(final_1['판매월'])

# '판매월'과 'cluster'별로 '판매수량' 합계 계산
final_1 = final_1.groupby([final_1['판매월'].dt.to_period('M'), 'cluster'])['판매수량'].sum().reset_index()

# '판매월'을 datetime 형식으로 다시 변환 (optional)
final_1['판매월'] = final_1['판매월'].dt.to_timestamp()

# 결과 확인
final_1

,판매월,cluster,판매수량
0,2021-01-01,1,3719.0
1,2021-01-01,2,2673.0
2,2021-01-01,3,1536.0
3,2021-01-01,4,880.0
4,2021-02-01,1,2654.0
...,...,...,...
139,2023-11-01,4,855.0
140,2023-12-01,1,3035.0
141,2023-12-01,2,1726.0
142,2023-12-01,3,881.0


---

## sarima

In [35]:
final_2 = final_1.copy()

In [23]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# final_2 데이터 로드
# final_2 = pd.read_csv('your_data.csv')
final_2['판매월'] = pd.to_datetime(final_2['판매월'])

# 학습 데이터 (2021년 1월 ~ 2023년 8월)
train = final_2[final_2['판매월'] < '2023-09-01']
# 테스트 데이터 (2023년 9월 ~ 2023년 12월)
test = final_2[final_2['판매월'] >= '2023-09-01']

In [24]:
# SARIMA 모델을 통한 예측 함수
def sarima_model(train_data, test_data, cluster):
    # 특정 cluster 데이터
    train_cluster = train_data[train_data['cluster'] == cluster]
    test_cluster = test_data[test_data['cluster'] == cluster]
    
    # 판매월을 인덱스로 설정
    train_cluster = train_cluster.set_index('판매월')['판매수량']
    test_cluster = test_cluster.set_index('판매월')['판매수량']
    
    # SARIMA 모델 학습 (기본 파라미터 사용)
    sarima_model = SARIMAX(train_cluster, order=(1,1,1), seasonal_order=(1,1,1,12))
    sarima_fit = sarima_model.fit(disp=False)
    
    # 예측 (2023년 9월 ~ 12월)
    pred = sarima_fit.predict(start=len(train_cluster), end=len(train_cluster) + len(test_cluster) - 1)
    
    return test_cluster, pred

In [25]:
# 모든 클러스터에 대해 SARIMA 예측 수행
clusters = final_2['cluster'].unique()
sarima_results = pd.DataFrame()

for cluster in clusters:
    actual, pred = sarima_model(train, test, cluster)
    
    # 결과를 데이터프레임에 저장
    temp_df = pd.DataFrame({'판매월': actual.index, 
                            'cluster': cluster, 
                            '실제 판매수량': actual.values, 
                            '예측 판매수량': pred.values})
    
    sarima_results = pd.concat([sarima_results, temp_df], axis=0)

In [26]:
# 월별로 실제값과 예측값 합계 계산
sarima_monthly = sarima_results.groupby('판매월').agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
sarima_monthly

,판매월,실제 판매수량,예측 판매수량
0,2023-09-01,6598.0,6202.511877
1,2023-10-01,7522.0,6375.089531
2,2023-11-01,6795.0,8153.579888
3,2023-12-01,6848.0,7351.825925


In [27]:
# MAPE 계산
mape = mean_absolute_percentage_error(sarima_monthly['실제 판매수량'], sarima_monthly['예측 판매수량']) * 100
print(f"최종 MAPE (SARIMA): {mape}")

최종 MAPE (SARIMA): 12.148140737990573


---

## sarima + gridsearchCV

In [38]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

In [36]:
# 데이터 준비
final_2['판매월'] = pd.to_datetime(final_2['판매월'])
train = final_2[final_2['판매월'] < '2023-09-01']
test = final_2[final_2['판매월'] >= '2023-09-01']

# SARIMA 파라미터 그리드 정의
param_grid = {
    'order': [(1,1,1), (2,1,2)],
    'seasonal_order': [(1,1,1,12), (1,1,2,12)]
}

# SARIMA 모델 학습 및 GridSearchCV
def sarima_gridsearch(train_data, test_data, cluster, param_grid):
    best_mape = float('inf')
    best_params = None
    best_model = None
    
    # 특정 클러스터 데이터
    train_cluster = train_data[train_data['cluster'] == cluster]
    test_cluster = test_data[test_data['cluster'] == cluster]
    
    train_cluster = train_cluster.set_index('판매월')['판매수량']
    test_cluster = test_cluster.set_index('판매월')['판매수량']
    
    # 파라미터 그리드를 순회하며 SARIMA 모델 학습
    for params in ParameterGrid(param_grid):
        try:
            model = SARIMAX(train_cluster, order=params['order'], seasonal_order=params['seasonal_order'])
            model_fit = model.fit(disp=False)
            
            pred = model_fit.predict(start=len(train_cluster), end=len(train_cluster) + len(test_cluster) - 1)
            mape = mean_absolute_percentage_error(test_cluster, pred) * 100
            
            # 최적 파라미터 및 모델 저장
            if mape < best_mape:
                best_mape = mape
                best_params = params
                best_model = model_fit
                
        except Exception as e:
            print(f"Error with parameters {params}: {e}")
    
    return best_model, best_params, best_mape

# 클러스터 1에 대해 GridSearch 적용
best_model, best_params, best_mape = sarima_gridsearch(train, test, 1, param_grid)

# 예측 수행
pred = best_model.predict(start=len(train[train['cluster'] == 1].set_index('판매월')['판매수량']),
                          end=len(train[train['cluster'] == 1].set_index('판매월')['판매수량']) + len(test[test['cluster'] == 1]) - 1)

In [37]:
# 결과 출력
print(f"최적 파라미터: {best_params}")
print(f"최종 MAPE (GridSearch): {best_mape:.2f}%")

최적 파라미터: {'order': (2, 1, 2), 'seasonal_order': (1, 1, 1, 12)}
최종 MAPE (GridSearch): 16.64%


---

## ensemble(randomforest, xgboost, lgbm) + gridsearchCV

In [64]:
final_2 = final_1.copy()

In [65]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [66]:
# 학습 및 테스트 데이터 준비 (동일)
train = final_2[final_2['판매월'] < '2023-09-01']
test = final_2[final_2['판매월'] >= '2023-09-01']

In [67]:
# 클러스터별 데이터 준비 함수
def prepare_data(cluster):
    train_cluster = train[train['cluster'] == cluster]
    test_cluster = test[test['cluster'] == cluster]

    X_train = train_cluster[['판매월']].apply(lambda x: x.view('int64')).values
    y_train = train_cluster['판매수량'].values
    X_test = test_cluster[['판매월']].apply(lambda x: x.view('int64')).values
    y_test = test_cluster['판매수량'].values

    return X_train, y_train, X_test, y_test

In [68]:
# GridSearch 및 모델 학습 함수
def ensemble_model(train_data, test_data, cluster, model_type):
    X_train, y_train, X_test, y_test = prepare_data(cluster)
    
    if model_type == 'RandomForest':
        model = RandomForestRegressor()
        param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20]}
    elif model_type == 'XGBoost':
        model = XGBRegressor()
        param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20], 'learning_rate': [0.01, 0.1]}
    elif model_type == 'LGBM':
        model = LGBMRegressor()
        param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20], 'learning_rate': [0.01, 0.1]}
    
    # GridSearchCV로 모델 최적화
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_absolute_percentage_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    # 최적 모델로 예측
    best_model = grid_search.best_estimator_
    pred = best_model.predict(X_test)

    return y_test, pred

In [58]:
# 클러스터별 앙상블 모델 적용
ensemble_results = pd.DataFrame()
models = ['RandomForest', 'XGBoost', 'LGBM']  # 사용할 모델 목록

for model_type in models:  # 각 모델에 대해 반복
    for cluster in final_2['cluster'].unique():
        actual, pred = ensemble_model(train, test, cluster, model_type)
        
        # 결과 저장
        temp_df = pd.DataFrame({
            '판매월': test[test['cluster'] == cluster]['판매월'],
            'cluster': cluster,
            '실제 판매수량': actual,
            '예측 판매수량': pred,
            '모델': model_type  # 어떤 모델에서 나온 예측인지 기록
        })
        
        ensemble_results = pd.concat([ensemble_results, temp_df], axis=0)

# 월별로 실제값과 예측값 합계 계산
ensemble_monthly = ensemble_results.groupby(['판매월', '모델']).agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
ensemble_monthly


[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2973.428571
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 22, number of used features: 0
[LightGBM] [Info] Start training from score 3091.409091
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2804.619048
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2973.428571
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2804.619048
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

...

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 32, number of used features: 0
[LightGBM] [Info] Start training from score 1079.312500
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

,판매월,모델,실제 판매수량,예측 판매수량
0,2023-09-01,LGBM,6598.0,7094.468750
1,2023-09-01,RandomForest,6598.0,6789.560000
2,2023-09-01,XGBoost,6598.0,6948.767456
3,2023-10-01,LGBM,7522.0,7094.468750
4,2023-10-01,RandomForest,7522.0,6789.560000
5,2023-10-01,XGBoost,7522.0,6948.767456
6,2023-11-01,LGBM,6795.0,7094.468750
7,2023-11-01,RandomForest,6795.0,6789.560000
8,2023-11-01,XGBoost,6795.0,6948.767456
9,2023-12-01,LGBM,6848.0,7094.468750


세 개 앙상블 한 거

In [59]:
# MAPE 계산
mape_ensemble = mean_absolute_percentage_error(ensemble_monthly['실제 판매수량'], ensemble_monthly['예측 판매수량']) * 100
print(f"최종 MAPE (Ensemble {models}): {mape_ensemble}")

최종 MAPE (Ensemble LGBM): 4.288342814851961


---

## RandomForest 단일모델

In [32]:
# 클러스터별로 Ensemble 모델 적용
ensemble_results = pd.DataFrame()
model_type = 'RandomForest'

for cluster in final_2['cluster'].unique():
    actual, pred = ensemble_model(train, test, cluster, model_type)
    
    # 결과 저장
    temp_df = pd.DataFrame({
        '판매월': test[test['cluster'] == cluster]['판매월'],
        'cluster': cluster,
        '실제 판매수량': actual,
        '예측 판매수량': pred
    })
    
    ensemble_results = pd.concat([ensemble_results, temp_df], axis=0)

# 월별로 실제값과 예측값 합계 계산
ensemble_monthly = ensemble_results.groupby('판매월').agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
ensemble_monthly

,판매월,실제 판매수량,예측 판매수량
0,2023-09-01,6598.0,6805.47
1,2023-10-01,7522.0,6805.47
2,2023-11-01,6795.0,6805.47
3,2023-12-01,6848.0,6805.47


In [33]:
# MAPE 계산
mape_ensemble = mean_absolute_percentage_error(ensemble_monthly['실제 판매수량'], ensemble_monthly['예측 판매수량']) * 100
print(f"최종 MAPE (Ensemble {model_type}): {mape_ensemble}")

최종 MAPE (Ensemble RandomForest): 3.361342462406325


---

## XGBoost 단일모델

In [69]:
# 클러스터별로 Ensemble 모델 적용
ensemble_results = pd.DataFrame()
model_type = 'XGBoost'

for cluster in final_2['cluster'].unique():
    actual, pred = ensemble_model(train, test, cluster, model_type)
    
    # 결과 저장
    temp_df = pd.DataFrame({
        '판매월': test[test['cluster'] == cluster]['판매월'],
        'cluster': cluster,
        '실제 판매수량': actual,
        '예측 판매수량': pred
    })
    
    ensemble_results = pd.concat([ensemble_results, temp_df], axis=0)

# 월별로 실제값과 예측값 합계 계산
ensemble_monthly = ensemble_results.groupby('판매월').agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
ensemble_monthly

,판매월,실제 판매수량,예측 판매수량
0,2023-09-01,6598.0,6948.767578
1,2023-10-01,7522.0,6948.767578
2,2023-11-01,6795.0,6948.767578
3,2023-12-01,6848.0,6948.767578


In [70]:
# MAPE 계산
mape_ensemble = mean_absolute_percentage_error(ensemble_monthly['실제 판매수량'], ensemble_monthly['예측 판매수량']) * 100
print(f"최종 MAPE (Ensemble {model_type}): {mape_ensemble}")

최종 MAPE (Ensemble XGBoost): 4.16786426372717


---

## LGBM 단일모델

In [71]:
# 클러스터별로 Ensemble 모델 적용
ensemble_results = pd.DataFrame()
model_type = 'LGBM'

for cluster in final_2['cluster'].unique():
    actual, pred = ensemble_model(train, test, cluster, model_type)
    
    # 결과 저장
    temp_df = pd.DataFrame({
        '판매월': test[test['cluster'] == cluster]['판매월'],
        'cluster': cluster,
        '실제 판매수량': actual,
        '예측 판매수량': pred
    })
    
    ensemble_results = pd.concat([ensemble_results, temp_df], axis=0)

# 월별로 실제값과 예측값 합계 계산
ensemble_monthly = ensemble_results.groupby('판매월').agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
ensemble_monthly

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2973.428571
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 22, number of used features: 0
[LightGBM] [Info] Start training from score 3091.409091
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2804.619048
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 22, number of used features: 0
[LightGBM] [Info] Start training from score 3091.409091
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 21, number of used features: 0
[LightGBM] [Info] Start training from score 2804.619048
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

...

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 32, number of used features: 0
[LightGBM] [Info] Start training from score 1079.312500
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

,판매월,실제 판매수량,예측 판매수량
0,2023-09-01,6598.0,7094.46875
1,2023-10-01,7522.0,7094.46875
2,2023-11-01,6795.0,7094.46875
3,2023-12-01,6848.0,7094.46875


In [72]:
# MAPE 계산
mape_ensemble = mean_absolute_percentage_error(ensemble_monthly['실제 판매수량'], ensemble_monthly['예측 판매수량']) * 100
print(f"최종 MAPE (Ensemble {model_type}): {mape_ensemble}")

최종 MAPE (Ensemble LGBM): 5.303651468070618


---

## 단일모델 코드 합친 거

In [60]:
# 클러스터별로 단일 모델 적용
results = pd.DataFrame()

# 모델 리스트
models = {
    'RandomForest': RandomForestRegressor(),
    'XGBoost': XGBRegressor(),
    'LGBM': LGBMRegressor()
}

# 각 모델에 대해 반복
for model_name, model in models.items():
    for cluster in final_2['cluster'].unique():
        X_train, y_train, X_test, y_test = prepare_data(cluster)
        
        # 모델 학습
        model.fit(X_train, y_train)
        
        # 예측
        pred = model.predict(X_test)

        # 결과 저장
        temp_df = pd.DataFrame({
            '판매월': test[test['cluster'] == cluster]['판매월'],
            'cluster': cluster,
            '모델': model_name,
            '실제 판매수량': y_test,
            '예측 판매수량': pred
        })
        
        results = pd.concat([results, temp_df], axis=0)

# 월별로 실제값과 예측값 합계 계산
monthly_results = results.groupby(['판매월', '모델']).agg({'실제 판매수량': 'sum', '예측 판매수량': 'sum'}).reset_index()

# 예측 결과 출력
monthly_results

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 32, number of used features: 0
[LightGBM] [Info] Start training from score 2958.593750
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the s

,판매월,모델,실제 판매수량,예측 판매수량
0,2023-09-01,LGBM,6598.0,7094.46875
1,2023-09-01,RandomForest,6598.0,6677.83000
2,2023-09-01,XGBoost,6598.0,6858.00000
3,2023-10-01,LGBM,7522.0,7094.46875
4,2023-10-01,RandomForest,7522.0,6677.83000
5,2023-10-01,XGBoost,7522.0,6858.00000
6,2023-11-01,LGBM,6795.0,7094.46875
7,2023-11-01,RandomForest,6795.0,6677.83000
8,2023-11-01,XGBoost,6795.0,6858.00000
9,2023-12-01,LGBM,6848.0,7094.46875


In [61]:
monthly_results

,판매월,모델,실제 판매수량,예측 판매수량
0,2023-09-01,LGBM,6598.0,7094.46875
1,2023-09-01,RandomForest,6598.0,6677.83000
2,2023-09-01,XGBoost,6598.0,6858.00000
3,2023-10-01,LGBM,7522.0,7094.46875
4,2023-10-01,RandomForest,7522.0,6677.83000
5,2023-10-01,XGBoost,7522.0,6858.00000
6,2023-11-01,LGBM,6795.0,7094.46875
7,2023-11-01,RandomForest,6795.0,6677.83000
8,2023-11-01,XGBoost,6795.0,6858.00000
9,2023-12-01,LGBM,6848.0,7094.46875


In [62]:
# MAPE 계산 함수
def calculate_mape(actual, predicted):
    return (abs(actual - predicted) / actual).mean() * 100

# 모델별 MAPE 계산
mape_results = monthly_results.groupby('모델').apply(
    lambda x: calculate_mape(x['실제 판매수량'], x['예측 판매수량'])
).reset_index()

# 컬럼 이름 변경
mape_results.columns = ['모델', 'MAPE']

# 결과 출력
mape_results

,모델,MAPE
0,LGBM,5.303651
1,RandomForest,4.160477
2,XGBoost,3.460302


[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

...

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more 

---

In [49]:
final_2 = final_1.copy()